In [1]:
import pandas as pd

orders_raw = pd.read_excel("pandas_cleaning.xlsx", sheet_name="Orders_Raw")

In [5]:
#print(orders_raw)

column_titles = orders_raw.columns

for title in column_titles:
    if ' ' in title:
        orders_raw.rename({title: title.replace(' ','_')}, inplace = True)
print(orders_raw.columns)

Index(['order_id', 'line_number', 'customer_id', 'customer_name',
       'customer_email', 'customer_phone', 'order_timestamp_raw',
       'ship_timestamp_raw', 'record_updated_at_raw', 'order_status',
       'channel', 'source_system', 'sku', 'product_name', 'category',
       'quantity_raw', 'unit_price_raw', 'discount_raw', 'shipping_cost_raw',
       'reported_total_raw', 'currency', 'country', 'city', 'postal_code',
       'payment_method', 'is_gift', 'coupon_code'],
      dtype='str')


In [ ]:
# print(len(orders_raw))
# print("\n")
# print(orders_raw.dtypes)
# print("\n")
# print(orders_raw.isna().sum()) #verify nulls in columns
# print(orders_raw.nunique())
# print(orders_raw.head(5))
# print(orders_raw[orders_raw.duplicated(subset=['order_id', 'line_number'])])


In [6]:
print(orders_raw.dtypes)

order_id                    str
line_number               int64
customer_id                 str
customer_name               str
customer_email              str
customer_phone              str
order_timestamp_raw      object
ship_timestamp_raw       object
record_updated_at_raw    object
order_status                str
channel                     str
source_system               str
sku                         str
product_name                str
category                    str
quantity_raw                str
unit_price_raw              str
discount_raw                str
shipping_cost_raw           str
reported_total_raw          str
currency                    str
country                     str
city                        str
postal_code                 str
payment_method              str
is_gift                     str
coupon_code                 str
dtype: object


In [7]:
print(len(orders_raw))
print("\n")
#print(orders_raw.dtypes)
#print("\n")
#print(orders_raw.isna().sum()) #verify nulls in columns
print(orders_raw[orders_raw.duplicated(subset=['order_id', 'line_number'])]) #randurile duplicate

10000


        order_id  line_number  customer_id   customer_name  \
1281  ORD-103633            1    CUS-00671  Olivia Dumitru   
1406  ORD-103336            1   cus-01142      Zofia Costa   
1415  ORD-103897            1    CUS-00078     Isla Nistor   
1985  ORD-102506            1    CUS-02437      Emma Nowak   
2056  ORD-100581            1    CUS-01635     Noah Jansen   
...          ...          ...          ...             ...   
9899  ORD-101909            1   cus-02222     CLAIRE SMITH   
9912  ORD-100642            1    CUS-01510      Radu Marin   
9914  ORD-100412            1    CUS-01908     Radu Brown    
9920  ORD-104395            1    CUS-02371  Victor Ionescu   
9990  ORD-101952            1    CUS-02316   Andrei Jansen   

                       customer_email customer_phone  order_timestamp_raw  \
1281   olivia.dumitru.671@example.com      486313649  03-19-2025 03:46 AM   
1406     zofia.costa.1142@example.com      341043499         45951.881296   
1415           i

In [ ]:
# 1. Load Orders_Raw with every raw business field preserved as text
# Standardise column names to snake_case
# Produce a compact profile with row count, data type, missing count, placeholder-missing count, unique count, sample values and deep memory usage
# Report exact duplicate rows and duplicate (order_id, line_number) keys, but do not remove them yet

# 2. Create reusable text-cleaning rules for whitespace, repeated spaces, case and known null markers
# Apply them to customer contact fields and categorical columns
# Use a canonical casing policy appropriate to each field: upper case for codes, lower case for emails and a consistent display form for names and labels

def clean_func(datafr:pd.DataFrame,nume_coloana:str):

    return_datafr = datafr.copy()
    return_datafr[nume_coloana] = return_datafr[nume_coloana].str.strip()
    return_datafr[nume_coloana]= return_datafr[nume_coloana].str.upper().replace(['N/A','NONE',' ','UNKNOWN','NOT AVAILABLE','MISSING'],pd.NA)

    if nume_coloana in ['source_system','customer_id','coupon_code','postal_code','sku','payment_method','channel','currency', 'country']:
        return_datafr[nume_coloana] = return_datafr[nume_coloana].str.upper()

    if nume_coloana == 'customer_email':
        return_datafr[nume_coloana]=return_datafr[nume_coloana].str.lower()

    return return_datafr[[nume_coloana]]

dfgol = orders_raw[['order_id','line_number']]
for col in orders_raw.columns:
    if col != 'order_id' and col != 'line_number':
        dfgol=dfgol.join(clean_func(orders_raw,col))

print(dfgol)

        order_id  line_number customer_id   customer_name  \
0     ORD-104276            1   CUS-02157      NOAH ROSSI   
1     ORD-102855            3   CUS-00666     EMMA WILSON   
2     ORD-101906            1   CUS-02281    RADU LEFEVRE   
3     ORD-100005            1   CUS-00633     EVA BIANCHI   
4     ORD-104162            1   CUS-00525  MIHAI KOWALSKI   
...          ...          ...         ...             ...   
9995  ORD-100716            1   CUS-01782   GIULIA MARTIN   
9996  ORD-102168            1   CUS-01356    LUCAS WILSON   
9997  ORD-100137            1   CUS-00686    IOANA GARCIA   
9998  ORD-100312            1   CUS-00449    MIHAI GARCIA   
9999  ORD-101276            2   CUS-01652   LUCAS BIANCHI   

                      customer_email customer_phone order_timestamp_raw  \
0        noah.rossi.2157@example.com      339081284    02/09/2025 04:01   
1        emma.wilson.666@example.com      406274054    2025/10/12 12:40   
2      radu.lefevre.2281@example.com      

In [ ]:
# 3. Convert quantity_raw, unit_price_raw, discount_raw, shipping_cost_raw and reported_total_raw to nullable numeric columns
# Handle currency symbols and codes, decimal commas, US and European thousands separators, percentages, surrounding spaces and accounting-style parentheses
# Separate parse failures from genuine missing values
# Typed numeric columns and a rejected-values table containing the original value, column and reason

# X (123.45) -> -123.45
# X ce nu este valoare numerica (RON, $...) -> except [^0-9%,.] (scoate si spatiile automat)

# daca avem % -> il scriem in procent ( /100)
    #daca n-avem indeajuns de multe cifre ca sa punem punctul (ex 1% -> 0.01 sau 12% -> 0.12)
# X , -> . 
# 1.031,2 -> 1,031.2 (mai sunt si 1.031,233)
    # X daca sunt cel putin 2 separatoare -> ne dam seama dupa primul si ultimul -> EU .  , / US ,  .
    # X daca e un singur separator si nu sunt 3 cifre dupa separator => separatorul e zecimal
    # X daca are fix 3 cifre we take the hit


print(orders_raw[["quantity_raw"]])

rejected_values = pd.DataFrame(columns = ["original_value", "column_name", "reason"])
orders_raw_copy = orders_raw.copy()

for column in orders_raw_copy.columns:
    if column in ["quantity_raw", "unit_price_raw", "discount_raw", "shipping_cost_raw", "reported_total_raw"]:
        orders_raw_copy[column] = orders_raw_copy[column].str.replace("(", "-").str.replace(")", "")
        orders_raw_copy[column] = orders_raw_copy[column].str.replace(r"[^0-9%,.]", "", regex = True)
        orders_raw_copy[column] = orders_raw_copy[column].str.replace(r",(\d+)$", r".\g<1>", regex = True).str.replace(r"\.(\d{3})\.", r",\g<1>,", regex = True).str.replace(r",(\d+)$", r".\g<1>", regex = True).str.replace(",", "")
        orders_raw_copy[column] = orders_raw_copy[column].str.replace(r"(.+)(\d{2})\.(\d+)%", r"\g<1>.\g<2>\g<3>", regex = True).str.replace(r"(\d{1})\.(\d+)%", r"0.0\g<1>\g<2>", regex = True).str.replace(r"(.+)(\d{2})%$", r"0\g<1>.\g<2>", regex = True).str.replace(r"^(\d{1})%$", r"0.0\g<1>", regex = True)
        orders_raw_copy[column] = pd.to_numeric(orders_raw_copy[column], errors = "coerce")
print(orders_raw_copy)
print("\n")

for column in orders_raw.columns: 
    if column in ["quantity_raw", "unit_price_raw", "discount_raw", "shipping_cost_raw", "reported_total_raw"]:
        column_value = orders_raw[orders_raw_copy[column].isna() & orders_raw[column].notna()][column]
        rejected_values = pd.concat([rejected_values,pd.DataFrame({"original_value":list(column_value.values), "column_name": column, "reason": "Unable to parse string"})])
print(rejected_values)

     quantity_raw
0           1 pcs
1               3
2               5
3               2
4               1
...           ...
9995            1
9996            1
9997            1
9998            1
9999            1

[10000 rows x 1 columns]
        order_id  line_number customer_id   customer_name  \
0     ORD-104276            1   CUS-02157      Noah Rossi   
1     ORD-102855            3   CUS-00666     Emma Wilson   
2     ORD-101906            1   CUS-02281    Radu Lefevre   
3     ORD-100005            1   CUS-00633     Eva Bianchi   
4     ORD-104162            1   CUS-00525  Mihai Kowalski   
...          ...          ...         ...             ...   
9995  ORD-100716            1   CUS-01782   Giulia Martin   
9996  ORD-102168            1   CUS-01356    lucas wilson   
9997  ORD-100137            1   CUS-00686    Ioana Garcia   
9998  ORD-100312            1   CUS-00449    Mihai Garcia   
9999  ORD-101276            2   CUS-01652   Lucas Bianchi   

                      cus

In [10]:
print(orders_raw_copy.dtypes)

order_id                     str
line_number                int64
customer_id                  str
customer_name                str
customer_email               str
customer_phone               str
order_timestamp_raw       object
ship_timestamp_raw        object
record_updated_at_raw     object
order_status                 str
channel                      str
source_system                str
sku                          str
product_name                 str
category                     str
quantity_raw             float64
unit_price_raw           float64
discount_raw             float64
shipping_cost_raw        float64
reported_total_raw       float64
currency                     str
country                      str
city                         str
postal_code                  str
payment_method               str
is_gift                      str
coupon_code                  str
dtype: object


In [ ]:
# 4. Use Source_System_Ref to parse order_timestamp_raw, ship_timestamp_raw and record_updated_at_raw according to each source system
# Localise naive values to the declared source timezone, convert everything to UTC and retain parse-failure flags
# Detect ship times before order time, updates before order time and shipping delays over 30 days

import pandas as pd

orders_raw = pd.read_excel('pandas_cleaning.xlsx', sheet_name='Orders_Raw')

source_sys_ref = pd.read_excel('pandas_cleaning.xlsx', sheet_name='Source_System_Ref')

In [ ]:
def fct(order_val):
    #pd.to_datetime(order_val, errors= 'coerce')
    if order_val.tzinfo == None:
        val = order_val.tz_localize(tz= source_sys_ref['source_timezone'].iloc[ent], ambiguous= 'NaT', nonexistent = 'shift_forward').tz_convert(tz='UTC')
        
    else:
        val = order_val.tz_convert(tz='UTC')
    return val
        


orders_raw = pd.read_excel(r'pandas_data_cleaning_course_pack.xlsx',sheet_name='Orders_Raw')

source_sys_ref = pd.read_excel(r'pandas_data_cleaning_course_pack.xlsx',sheet_name='Source_System_Ref')
#print(source_sys_ref)
source_sys_ref['timestamp_pattern'] = source_sys_ref['timestamp_pattern'].str.replace('YYYY', '%Y').str.replace('MM', '%m').str.replace('DD', '%d').str.replace('hh', '%H').str.replace('HH', '%H').str.replace('mm', '%M').str.replace('AM', '%p').str.replace('ISO 8601 with Z', 'ISO8601')
temp = orders_raw.merge(source_sys_ref, on = 'source_system')
order_t_parsed = orders_raw.copy()

for ent in range(len(source_sys_ref['source_system'])):
    #print(source_sys_ref['source_system'].iloc[ent])
    #print(temp[temp['source_system']==source_sys_ref['source_system'].iloc[ent]]['order_timestamp_raw'])
    #print(source_sys_ref.loc[ent,['timestamp_pattern']].iloc[0])
    #print(list(order_t_parsed[order_t_parsed['source_system']==source_sys_ref['source_system'].iloc[ent]].index))
    if ent == len(source_sys_ref['source_system'])-1:
        order_t_parsed.loc[list(order_t_parsed[order_t_parsed['source_system']==source_sys_ref['source_system'].iloc[ent]].index), ['order_timestamp_raw']] = pd.to_datetime(pd.to_numeric(temp[temp['source_system']==source_sys_ref['source_system'].iloc[ent]]['order_timestamp_raw'], errors= 'coerce'), errors= 'coerce', unit= 'D', origin='1899-12-30')
        #print(order_t_parsed.loc[list(order_t_parsed[order_t_parsed['source_system']==source_sys_ref['source_system'].iloc[ent]].index), ['order_timestamp_raw']])
            
        order_t_parsed.loc[list(order_t_parsed[order_t_parsed['source_system']==source_sys_ref['source_system'].iloc[ent]].index), ['ship_timestamp_raw']] = pd.to_datetime(pd.to_numeric(temp[temp['source_system']==source_sys_ref['source_system'].iloc[ent]]['ship_timestamp_raw'], errors= 'coerce'), errors= 'coerce', unit= 'D', origin='1899-12-30')
            
        order_t_parsed.loc[list(order_t_parsed[order_t_parsed['source_system']==source_sys_ref['source_system'].iloc[ent]].index), ['record_updated_at_raw']] = pd.to_datetime(pd.to_numeric(temp[temp['source_system']==source_sys_ref['source_system'].iloc[ent]]['record_updated_at_raw'], errors= 'coerce'), errors= 'coerce', unit= 'D', origin='1899-12-30')
    
    else:        
        order_t_parsed.loc[list(order_t_parsed[order_t_parsed['source_system']==source_sys_ref['source_system'].iloc[ent]].index), ['order_timestamp_raw']] = pd.to_datetime(temp[temp['source_system']==source_sys_ref['source_system'].iloc[ent]]['order_timestamp_raw'], errors= 'coerce', format= source_sys_ref.loc[ent,['timestamp_pattern']].iloc[0]).apply(fct)
        #print(order_t_parsed.loc[list(order_t_parsed[order_t_parsed['source_system']==source_sys_ref['source_system'].iloc[ent]].index), ['order_timestamp_raw']])
    
        order_t_parsed.loc[list(order_t_parsed[order_t_parsed['source_system']==source_sys_ref['source_system'].iloc[ent]].index), ['ship_timestamp_raw']] = pd.to_datetime(temp[temp['source_system']==source_sys_ref['source_system'].iloc[ent]]['ship_timestamp_raw'], errors= 'coerce', format= source_sys_ref.loc[ent,['timestamp_pattern']].iloc[0]).apply(fct)
    
        order_t_parsed.loc[list(order_t_parsed[order_t_parsed['source_system']==source_sys_ref['source_system'].iloc[ent]].index), ['record_updated_at_raw']] = pd.to_datetime(temp[temp['source_system']==source_sys_ref['source_system'].iloc[ent]]['record_updated_at_raw'], errors= 'coerce', format= source_sys_ref.loc[ent,['timestamp_pattern']].iloc[0]).apply(fct)
    
    #print(order_t_parsed.loc[list(order_t_parsed[order_t_parsed['source_system']==source_sys_ref['source_system'].iloc[ent]].index), ['order_timestamp_raw']])

print('\n')
mask1 = (order_t_parsed['ship_timestamp_raw'] - order_t_parsed['order_timestamp_raw'])<pd.Timedelta(nanoseconds=0)
print(order_t_parsed[mask1][['ship_timestamp_raw', 'order_timestamp_raw']])

print('\n')
mask2 = (order_t_parsed['record_updated_at_raw'] - order_t_parsed['order_timestamp_raw'])<pd.Timedelta(nanoseconds=0)
print(order_t_parsed[mask2][['record_updated_at_raw', 'order_timestamp_raw']])
    
print('\n')
mask3 = (order_t_parsed['ship_timestamp_raw'] - order_t_parsed['order_timestamp_raw'])>pd.Timedelta(days=30)
print(order_t_parsed[mask3][['ship_timestamp_raw', 'order_timestamp_raw']])

print('\n')
print(order_t_parsed[~(mask1 | mask2 | mask3)])
